# State Spending in More Detail
The state-spending page groups the budget into broad areas. This page opens those areas up: first by **type of spending**, then by **department** within each major area.

**How to read it**
- **Type of spending:** *State operations* is what state departments spend to run themselves (mostly pay and operating costs); *local assistance* is money the state sends on to local governments, schools and individuals (for example Medi-Cal payments to health providers and aid to school districts); *capital outlay* is buildings and construction.
- **Departments change names.** The Legislative Analyst's Office renames and reorganizes departments, many of them in FY2013-14. Successors of the same function are chained (for example Department of Health Services and then the Department of Health Care Services) in `ca_taxes.state.DEPARTMENT_CHAINS`. Splits and mergers, such as Mental Health moving into State Hospitals, are not chained, so some lines still begin or end in a year when a department was reorganized.
- All funds (general, special, federal and bond). Federal money passes through the state in several areas; see the federal page.
- Years after FY2025 are budget estimates (shaded).

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from ca_taxes.io import PROCESSED
from ca_taxes.deflate import add_views, base_fy, LAST_ACTUAL_FY
from ca_taxes.style import stacked_area_toggle, line_toggle, base_layout, PALETTE, SOURCE_NOTE
pio.renderers.default = "plotly_mimetype"
from ca_taxes.state import harmonize_department
e = pd.read_csv(PROCESSED / "state_expenditures.csv")
e["department"] = e.department.map(harmonize_department)
d = add_views(e.groupby(["fy", "exp_type"], as_index=False).amount_k.sum())
stacked_area_toggle(d, "exp_type", "State spending by type: operations, local assistance, capital", SOURCE_NOTE,
                    order=["Local Assistance", "State Operations", "Capital Outlay", "Unclassified"]).show()

## Inside each area
Top departments in each major area (up to 8, the rest grouped). Use the buttons for nominal, inflation-adjusted, per-person and share views.

In [2]:
AREAS = ["Health and Human Services", "K-12 Education", "Higher Education", "Transportation", "Corrections and Rehabilitation", "Labor and Workforce Development",
         "Natural Resources", "Environmental Protection", "Debt service (GO bonds)"]
for area in AREAS:
    x = e[e.area == area].groupby(["fy", "department"], as_index=False).amount_k.sum()
    top = x.groupby("department").amount_k.sum().sort_values(ascending=False).head(8).index
    x["department"] = x.department.where(x.department.isin(top), "All other departments")
    stacked_area_toggle(add_views(x.groupby(["fy", "department"], as_index=False).amount_k.sum()), "department", f"{area}: spending by department",
                        SOURCE_NOTE + ". Departments renamed over time are chained.").show()